In [1]:
from neo4j import GraphDatabase
uri = 'neo4j://localhost:7687'
user = 'neo4j'
password = 'password'

driver = GraphDatabase.driver(uri, auth=(user,password))

1. Найти самый старый фильм и самый новый фильм - вывести их названия по 1 шт (2 запроса)


In [7]:
def get_oldest_movie(tx):
    result = tx.run("""
            MATCH (m:Movie)
            RETURN m.title as oldest_movie, m.released
            ORDER BY m.released ASC
            LIMIT 1
    """
    )
    return [mov for mov in result]

def get_newest_movie(tx):
    result = tx.run("""
            MATCH (m:Movie)
            RETURN m.title as newest_movie, m.released
            ORDER BY m.released DESC
            LIMIT 1
    """
    )
    return [mov for mov in result]

with driver.session() as session:
    oldest_movie = session.execute_read(get_oldest_movie)
    for m in oldest_movie:
        print(f'{m["oldest_movie"]}-{m["m.released"]}')

with driver.session() as session:
    newest_movie = session.execute_read(get_newest_movie)
    for m in newest_movie:
        print(f'{m["newest_movie"]}-{m["m.released"]}')


One Flew Over the Cuckoo's Nest-1975
Cloud Atlas-2012


2. Получить среднее количество актёров на фильм

In [11]:
def get_avg_actors(tx):
    result = tx.run("""
            MATCH (m:Movie)<-[:ACTED_IN]-(actor:Person)
            WITH m, COUNT(actor) as actors_count
            RETURN AVG(actors_count) as avg_actors
    """
    )
    return result.single()["avg_actors"]


with driver.session() as session:
    avg_actors = session.execute_read(get_avg_actors)
    print(round(avg_actors,1))

4.5


3. Группировка фильмов по годам и подсчёт количества фильмов в каждом году

In [13]:
def get_movie_count(tx):
    result = tx.run("""
            MATCH (m:Movie)
            WITH m.released as year_released, COUNT(m) as movie_count
            RETURN year_released, movie_count
            ORDER BY year_released
    """
    )
    return [mov for mov in result]


with driver.session() as session:
    movie_count = session.execute_read(get_movie_count)
    for m in movie_count:
        print(f'{m["year_released"]}-{m["movie_count"]}\n')

1975-1

1986-2

1990-1

1992-4

1993-1

1995-2

1996-3

1997-2

1998-3

1999-4

2000-3

2003-3

2004-1

2006-3

2007-1

2008-2

2009-1

2012-1



In [14]:
driver.close()